# 机组排班（Beasley-Cao csp50）—— Benders（选列主问题 + 覆盖 LP 子问题）

## 问题定义

50 个任务 $i$（固定起止时间 $s_i,f_i$）；时间上限 $T=480$；173 条转移弧 $(i,j,c_{ij})$。
一个 crew 的任务序列须逐对由弧连接（弧列表已编码时间兼容性）且**跨度** $f_{last}-s_{first}\le 480$。
目标：**最少 crew 数，其次最小总转移成本**。集合覆盖模型：

$$\min_x \sum_{p\in P} c_p x_p \quad \text{s.t.}\quad \sum_{p\in P} a_{ip}x_p \ge 1\ (\forall i),\quad \sum_{p\in P} x_p \le K,\quad x_p\in\{0,1\}$$

列 $p$ = 一条可行 crew 调度（任务序列），$c_p$ = 序列转移成本之和。
**基准最优**：27 crew、成本 3139（本家族 01 直接模型证明，K=26 不可行）。


In [1]:
# -*- coding: utf-8 -*-
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import math, time, datetime
from ortools.math_opt.python import mathopt
from ortools.sat.python import cp_model

DATA = "/mnt/d/exactTest/column-generation-testcases/crew_scheduling/csp50.txt"
_lines = open(DATA).read().splitlines()
N, T = map(int, _lines[0].split())
tasks = [None] + [tuple(map(int, l.split())) for l in _lines[1:1+N]]
arc_cost = {}
for l in _lines[1+N:]:
    i, j, c = map(int, l.split()); arc_cost[(i, j)] = c
K_MIN = 27
EPS = 1e-7
M_DUMMY = 10**6

def enumerate_pool():
    adj = {}
    for (i, j), c in arc_cost.items():
        adj.setdefault(i, []).append((j, c))
    paths = []
    for start in range(1, N+1):
        s0 = tasks[start][0]
        stack = [(start, [start], 0)]
        while stack:
            u, seq, c = stack.pop()
            paths.append((tuple(seq), c, tasks[u][1]-s0))
            for v, cv in adj.get(u, []):
                span = tasks[v][1] - s0
                if span <= T:
                    stack.append((v, seq+[v], c+cv))
    return paths

paths = enumerate_pool()
P = len(paths)
pseq = [p[0] for p in paths]
pcost = [p[1] for p in paths]
pmask = []
for s2 in pseq:
    m2 = 0
    for i in s2:
        m2 |= (1 << i)
    pmask.append(m2)


print(f"完整池: {P} 条可行路径（单任务 {sum(1 for s in pseq if len(s)==1)} / 双 {sum(1 for s in pseq if len(s)==2)} / 三 {sum(1 for s in pseq if len(s)==3)}）")
print(f"下界: 时长={math.ceil(sum(f-s for s,f in tasks[1:])/T)} | 已知最优: 27 crew / 3139（K=26 不可行）")


完整池: 266 条可行路径（单任务 50 / 双 173 / 三 43）
下界: 时长=14 | 已知最优: 27 crew / 3139（K=26 不可行）


## 方法：Benders（选列主问题 + 覆盖 LP 子问题）

- **主问题 MP（HIGHS MIP）**：$\min\theta$，$y_p\in\{0,1\}$，最优性割
  $\theta+\sum_p\lambda_p^k y_p\ge\sum_i\pi_i^k+K\mu^k$（列成本含在子问题中）。
- **子问题 SP(y)（GLOP LP）**：固定 y：$\min\sum c_px_p+M\sum s_i$，
  s.t. 覆盖 $+s_i\ge1$（$\pi_i\ge0$）、$\sum x_p\le K$（$\mu\le0$）、$0\le x_p\le y_p$（$\sigma_p\le0$）；
  对偶 $\lambda=-\sigma$ 生成割（弱对偶：对偶可行点对任意 y 有效）。
- 候选集 = 完整池（266 列）；整数修复 = 完整池 HIGHS。


In [2]:
def col_arcs(seq):
    a = []
    prev = 0
    for j in seq:
        a.append((prev, j))
        prev = j
    a.append((prev, N+1))
    return a

def solve_sp(yvec):
    sp = mathopt.Model()
    xv = [sp.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{p}") for p in range(P)]
    sv = [sp.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"s{i}") for i in range(1, N+1)]
    covs = []
    for i in range(1, N+1):
        covs.append(sp.add_linear_constraint(
            mathopt.fast_sum([xv[p] for p in range(P) if (pmask[p] >> i) & 1]) + sv[i-1] >= 1.0, name=f"c{i}"))
    veh = sp.add_linear_constraint(mathopt.fast_sum(xv) <= K_MIN, name="veh")
    xub = [sp.add_linear_constraint(xv[p] <= float(yvec[p]), name=f"xub{p}") for p in range(P)]
    sp.minimize(mathopt.fast_sum([pcost[p]*xv[p] for p in range(P)]) + M_DUMMY*mathopt.fast_sum(sv))
    res = mathopt.solve(sp, mathopt.SolverType.GLOP)
    assert res.termination.reason == mathopt.TerminationReason.OPTIMAL
    dv = res.dual_values()
    pi = [max(0.0, dv[covs[i-1]]) for i in range(1, N+1)]
    mu = dv[veh]
    alpha = sum(pi) + K_MIN*mu
    lam = [-dv[xub[p]] for p in range(P)]
    return alpha, lam, res.objective_value()

def solve_master(cuts):
    mp = mathopt.Model()
    yv = [mp.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"y{p}") for p in range(P)]
    theta = mp.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name="theta")
    for kk, (alpha, lam) in enumerate(cuts):
        mp.add_linear_constraint(theta + mathopt.fast_sum([lam[p]*yv[p] for p in range(P) if lam[p] != 0.0]) >= alpha, name=f"bcut{kk}")
    mp.minimize(theta)
    res = mathopt.solve(mp, mathopt.SolverType.HIGHS,
                        params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=20), enable_output=False))
    vals = res.variable_values(yv)
    ystar = [1 if vals[p] > 0.5 else 0 for p in range(P)]
    th = res.variable_values([theta])[0]
    return ystar, th, res.objective_value()

def run_benders(verbose=True):
    current = [1]*P
    cuts = []
    wall0 = time.time()
    for it in range(20):
        alpha, lam, sp_obj = solve_sp(current)
        cuts.append((alpha, lam))
        ystar, theta, mp_obj = solve_master(cuts)
        if verbose:
            print(f"iter {it+1}: SP={round(sp_obj,4)} | MP={round(mp_obj,4)} theta={round(theta,4)} | y选中 {sum(ystar)} | 割 {len(cuts)}")
        if ystar == current:
            break
        current = ystar
        if time.time()-wall0 > 110:
            break
    # 整数修复（完整池 HIGHS）
    m = mathopt.Model()
    x = [m.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"x{p}") for p in range(P)]
    for i in range(1, N+1):
        m.add_linear_constraint(mathopt.fast_sum([x[p] for p in range(P) if (pmask[p] >> i) & 1]) >= 1.0, name=f"c{i}")
    m.add_linear_constraint(mathopt.fast_sum(x) <= K_MIN, name="veh")
    m.minimize(mathopt.fast_sum([pcost[p]*x[p] for p in range(P)]))
    res = mathopt.solve(m, mathopt.SolverType.HIGHS,
                        params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=60), enable_output=False))
    routes = [pseq[p] for p in range(P) if res.variable_values()[x[p]] > 0.5]
    if verbose:
        print(f"整数修复: {res.termination.reason.name} obj={res.objective_value()} crew={len(routes)}")
        print(f"Benders 下界 {round(mp_obj,4)} vs 修复 {res.objective_value()} | gap {(res.objective_value()-mp_obj)/mp_obj*100 if mp_obj else 0:.4f}%")
    return dict(iterations=it+1, cuts=len(cuts), lb=mp_obj, ip=res.objective_value(),
                routes=routes, status=res.termination.reason.name)

res = run_benders(verbose=True)


iter 1: SP=3139.0 | MP=3139.0 theta=3139.0 | y选中 7 | 割 1
iter 2: SP=36000809.0 | MP=3139.0 theta=3139.0 | y选中 266 | 割 2
iter 3: SP=3139.0 | MP=3139.0 theta=3139.0 | y选中 266 | 割 3
整数修复: OPTIMAL obj=3139.0 crew=27
Benders 下界 3139.0 vs 修复 3139.0 | gap -0.0000%


## 运行结果与结论

见上方输出。结论：

- **Benders 3 轮 / 3 条对偶割收敛，下界 = 整数修复 = 3139（27 crew）⇒ 证明最优。**
- 基准最优值来源：本家族 01_direct 证明（K=26 不可行 + K=27 最优 3139）；
  各方法结果交叉一致。
